# Credit Card Fraud Anomaly Detection

This notebook builds a cost-sensitive classifier to detect fraudulent credit card transactions. Fraud datasets are typically highly imbalanced (e.g. < 1% fraud rate). We generate a synthetic dataset with features representing transactions, train a class-weighted Random Forest, and analyze performance using Precision-Recall (PR) curves.



In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, precision_recall_curve, auc, confusion_matrix

# Generate highly imbalanced dataset (99.2% genuine, 0.8% fraud)
X, y = make_classification(
    n_samples=2000, 
    n_features=10, 
    n_informative=8, 
    n_redundant=2, 
    weights=[0.992, 0.008],
    random_state=42
)

# Convert to DataFrame
feature_names = [f'V{i}' for i in range(1, 10)] + ['Amount']
df = pd.DataFrame(X, columns=feature_names)
# Rescale Amount column to realistic dollar values
df['Amount'] = np.abs(df['Amount'] * 85.0) + 5.0
df['Class'] = y

print(f"Class distribution:\n{df['Class'].value_counts(normalize=True)}")
df.head()



Class distribution:
Class
0    0.9885
1    0.0115
Name: proportion, dtype: float64


,V1,V2,V3,V4,V5,V6,V7,V8,V9,Amount,Class
0,-0.401097,1.249961,-4.061954,2.346665,-2.532279,-0.975788,-0.449764,-0.361649,0.094370,97.726683,0
1,-2.253033,0.221874,-3.631354,0.398132,-5.032429,-0.530143,-2.202510,2.870743,2.309523,103.049811,0
2,-5.085149,2.125224,-3.147242,1.264669,-5.274072,0.868469,0.118780,1.814647,0.779287,96.504848,0
3,-0.692036,-0.771921,-0.039825,-1.036969,-3.185746,0.062163,-0.860032,0.553504,1.974981,217.345688,0
4,0.068853,0.023027,-0.872126,1.087298,2.515877,0.512387,0.954428,-0.110373,0.645181,94.996397,0


## Cost-Sensitive Random Forest Training

We utilize a class-weighted Random Forest classifier (`class_weight='balanced'`) to penalize misclassifications of the rare fraud class, preventing the model from predicting the majority class exclusively.



In [2]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

# Fit Random Forest with balanced class weights
clf = RandomForestClassifier(
    n_estimators=150, 
    max_depth=7, 
    class_weight='balanced', 
    random_state=42
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_probs = clf.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Genuine", "Fraud"]))



Classification Report:
              precision    recall  f1-score   support

     Genuine       0.99      0.97      0.98       593
       Fraud       0.00      0.00      0.00         7

    accuracy                           0.96       600
   macro avg       0.49      0.49      0.49       600
weighted avg       0.98      0.96      0.97       600



## Precision-Recall Curve Visualization

For highly imbalanced datasets, the Receiver Operating Characteristic (ROC) curve can present an overly optimistic view. The Precision-Recall (PR) curve is much more informative for measuring model discrimination.



In [3]:
precision, recall, _ = precision_recall_curve(y_test, y_probs)
pr_auc = auc(recall, precision)

# Plot Precision-Recall using Plotly
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=recall, 
    y=precision, 
    mode='lines', 
    name=f'Weighted Random Forest (AUC = {pr_auc:.3f})',
    line=dict(color='#F43F5E', width=3)
))
fig.add_trace(go.Scatter(
    x=[0, 1], 
    y=[0.008, 0.008], 
    mode='lines', 
    name='Baseline No-Skill',
    line=dict(color='#64748B', width=1.5, dash='dash')
))
fig.update_layout(
    title=dict(text="Precision-Recall Curve (Highly Imbalanced Data)"),
    xaxis=dict(title=dict(text="Recall (Sensitivity)")),
    yaxis=dict(title=dict(text="Precision (Positive Predictive Value)")),
    margin=dict(l=40, r=40, t=50, b=40)
)
fig.show()
